# Pairs Trading (Cointegration)
- Tải 2 ticker, kiểm tra cointegration, z-score spread, backtest đơn giản


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import coint

tickers = ['XOM','CVX']
prices = yf.download(tickers, start='2015-01-01', progress=False)['Adj Close']
prices = prices.dropna()
score, pvalue, _ = coint(prices[tickers[0]], prices[tickers[1]])
print('p-value:', pvalue)

spread = prices[tickers[0]] - prices[tickers[1]]
z = (spread - spread.mean())/spread.std()
entry = 2
exit = 0
longs = z < -entry
shorts = z > entry
flat = (abs(z) < exit)
pos = np.where(longs, 1, np.where(shorts, -1, np.nan))
pos = pd.Series(pos, index=prices.index).ffill().fillna(0)

ret0 = prices[tickers[0]].pct_change().fillna(0)
ret1 = prices[tickers[1]].pct_change().fillna(0)
pair_ret = pos.shift(1).fillna(0) * (ret0 - ret1)

def perf(returns, freq=252):
    cum = (1+returns).prod()-1
    cagr = (1+cum)**(freq/len(returns))-1
    vol = returns.std()*np.sqrt(freq)
    sharpe = returns.mean()*freq/(returns.std()*np.sqrt(freq)) if vol!=0 else np.nan
    dd = (1+returns).cumprod()
    peak = dd.cummax()
    mdd = (dd/peak-1).min()
    return {'pvalue': pvalue, 'CAGR': cagr, 'Sharpe': sharpe, 'MaxDD': mdd}

perf(pair_ret)
